In [24]:
import joblib
import pandas as pd
import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier

from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import cross_val_score
import optuna
from sklearn.metrics import roc_auc_score

import sys
sys.path.append('../')
from src import *

In [8]:
X_train = joblib.load('../data/05/X_train_up.pkl')
X_valid = joblib.load('../data/05/X_valid_up.pkl')
X_test = joblib.load('../data/05/X_test_up.pkl')

y_train = joblib.load('../data/01/y_train.pkl')
y_valid = joblib.load('../data/01/y_valid.pkl')
y_test = joblib.load('../data/01/y_test.pkl')

selected_features_hand = joblib.load('../data/06/selected_features_hand.pkl')
selected_features_l1 = joblib.load('../data/06/selected_features_l1.pkl')

In [9]:
top_features = selected_features_hand
X_train_best = X_train[top_features]
X_valid_best = X_valid[top_features]

## Logistic regression

In [10]:
lr_base = LogisticRegression(
    penalty="l2",
    C=1.0,
    solver="lbfgs",
    max_iter=2000
)

lr_base.fit(X_train_best, y_train)
pred = lr_base.predict_proba(X_valid_best)[:,1]

print("Baseline GINI:", gini_score(y_valid, pred))

Baseline GINI: 0.4729197175448576


### GridSearch

In [11]:
param_grid = {
    "C": [0.01, 0.1, 1, 5, 10, 50],
    "solver": ["lbfgs", "liblinear"],
    "max_iter": [1000, 2000, 3000]
}

lr = LogisticRegression(penalty="l2")

grid = GridSearchCV(
    lr,
    param_grid,
    scoring="roc_auc",
    cv=3,
    n_jobs=-1,
    verbose=1
)

grid.fit(X_train_best, y_train)

print("Best params:", grid.best_params_)
print("Best CV AUC:", grid.best_score_)

Fitting 3 folds for each of 36 candidates, totalling 108 fits
Best params: {'C': 0.01, 'max_iter': 1000, 'solver': 'lbfgs'}
Best CV AUC: 0.7417675158337507


In [12]:
best_lr_grid = grid.best_estimator_
pred_val_grid_lr = best_lr_grid.predict_proba(X_valid_best)[:,1]
print("Grid GINI:", gini_score(y_valid, pred_val_grid_lr))

Grid GINI: 0.47096341606428727


### RandomSearch

In [13]:
param_dist = {
    "C": np.logspace(-3, 2, 50),
    "solver": ["lbfgs", "liblinear"],
    "max_iter": [1000, 2000, 3000]
}

rand = RandomizedSearchCV(
    LogisticRegression(penalty="l2"),
    param_distributions=param_dist,
    n_iter=30,
    scoring="roc_auc",
    cv=3,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

rand.fit(X_train_best, y_train)

print("Best params:", rand.best_params_)

Fitting 3 folds for each of 30 candidates, totalling 90 fits
Best params: {'solver': 'lbfgs', 'max_iter': 1000, 'C': np.float64(0.005179474679231213)}


In [14]:
best_lr_rand = rand.best_estimator_
pred_val_rand_lr = best_lr_rand.predict_proba(X_valid_best)[:,1]
print("RandomSearch GINI:", gini_score(y_valid, pred_val_rand_lr))

RandomSearch GINI: 0.4714970150504849


### Optuna

In [15]:
def objective(trial):
    C = trial.suggest_loguniform("C", 1e-3, 1e2)
    solver = trial.suggest_categorical("solver", ["lbfgs", "liblinear"])
    max_iter = trial.suggest_int("max_iter", 500, 4000)

    lr = LogisticRegression(
        penalty="l2",
        C=C,
        solver=solver,
        max_iter=max_iter
    )

    lr.fit(X_train_best, y_train)
    preds = lr.predict_proba(X_valid_best)[:,1]

    auc = roc_auc_score(y_valid, preds)
    return auc

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50)

print("Best trial:", study.best_trial.params)

[I 2026-01-25 17:58:48,196] A new study created in memory with name: no-name-10916d6d-912b-419c-8400-040d5c6ef67f
/tmp/ipykernel_323193/1613401812.py:2: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  C = trial.suggest_loguniform("C", 1e-3, 1e2)
[I 2026-01-25 17:58:48,441] Trial 0 finished with value: 0.6438982589919627 and parameters: {'C': 50.9483367830842, 'solver': 'liblinear', 'max_iter': 1712}. Best is trial 0 with value: 0.6438982589919627.
/tmp/ipykernel_323193/1613401812.py:2: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  C = trial.suggest_loguniform("C", 1e-3, 1e2)
[I 2026-01-25 17:58:48,527] Trial 1 finished with value: 0.7359892400591672 and parameters: {'C': 

Best trial: {'C': 88.85443904223031, 'solver': 'lbfgs', 'max_iter': 543}


In [16]:
best_params = study.best_trial.params

lr_optuna = LogisticRegression(
    penalty="l2",
    **best_params
)

lr_optuna.fit(X_train_best, y_train)
pred_val_opt_lr = lr_optuna.predict_proba(X_valid_best)[:,1]

print("Optuna GINI:", gini_score(y_valid, pred_val_opt_lr))


Optuna GINI: 0.47288082540371557


## Naive Bayes

### GridSearch

In [17]:
param_grid_nb = {
    "var_smoothing": [1e-9, 1e-8, 1e-7, 1e-6, 1e-5]
}

grid_nb = GridSearchCV(
    GaussianNB(),
    param_grid=param_grid_nb,
    scoring="roc_auc",
    cv=3,
    n_jobs=-1
)
grid_nb.fit(X_train_best, y_train)

print("Best GridSearch params (NB):", grid_nb.best_params_)
pred_val_grid_nb = grid_nb.predict_proba(X_valid_best)[:, 1]
print("Gini (GridSearch NB):", gini_score(y_valid, pred_val_grid_nb))

Best GridSearch params (NB): {'var_smoothing': 1e-05}
Gini (GridSearch NB): 0.4589168505959438


### RandomSearch

In [18]:
param_dist_nb = {
    "var_smoothing": np.logspace(-12, -3, 100)
}

random_nb = RandomizedSearchCV(
    GaussianNB(),
    param_distributions=param_dist_nb,
    n_iter=20,
    scoring="roc_auc",
    cv=3,
    n_jobs=-1,
    random_state=42
)
random_nb.fit(X_train[top_features], y_train)

pred_val_rand_nb = random_nb.predict_proba(X_valid[top_features])[:, 1]
print("Gini (RandomSearch NB):", gini_score(y_valid, pred_val_rand_nb))

Gini (RandomSearch NB): 0.4589190525993445


### Optuna

In [19]:
def objective_nb(trial):
    var_smoothing = trial.suggest_loguniform("var_smoothing", 1e-12, 1e-3)
    
    nb = GaussianNB(var_smoothing=var_smoothing)
    score = cross_val_score(nb, X_train[top_features], y_train, cv=3, scoring="roc_auc").mean()
    return score

study_nb = optuna.create_study(direction="maximize")
study_nb.optimize(objective_nb, n_trials=30)

best_params_nb = study_nb.best_params
print("Best Optuna params (NB):", best_params_nb)

nb_optuna = GaussianNB(**best_params_nb)
nb_optuna.fit(X_train[top_features], y_train)
pred_val_opt_nb = nb_optuna.predict_proba(X_valid[top_features])[:, 1]
print("Gini (Optuna NB):", gini_score(y_valid, pred_val_opt_nb))

[I 2026-01-25 17:58:58,590] A new study created in memory with name: no-name-72ea3cab-3c00-4b65-9ca6-e6a638b9bd8b
/tmp/ipykernel_323193/1685843492.py:2: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  var_smoothing = trial.suggest_loguniform("var_smoothing", 1e-12, 1e-3)
[I 2026-01-25 17:58:58,733] Trial 0 finished with value: 0.7325766927024953 and parameters: {'var_smoothing': 1.4596006451098473e-07}. Best is trial 0 with value: 0.7325766927024953.
/tmp/ipykernel_323193/1685843492.py:2: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  var_smoothing = trial.suggest_loguniform("var_smoothing", 1e-12, 1e-3)
[I 2026-01-25 17:58:58,865] Trial 1 finished with value: 0.7325766927

Best Optuna params (NB): {'var_smoothing': 0.0009969432369523836}
Gini (Optuna NB): 0.45893776962824906


## KNN

### GridSearch

In [20]:
param_grid_knn = {
    "n_neighbors": [3, 5, 7, 9, 11],
    "weights": ["uniform", "distance"],
    "metric": ["euclidean", "manhattan", "minkowski"]
}

grid_knn = GridSearchCV(
    KNeighborsClassifier(),
    param_grid=param_grid_knn,
    scoring="roc_auc",
    cv=3,
    n_jobs=-1
)

grid_knn.fit(X_train_best, y_train)
print("Best GridSearch params (KNN):", grid_knn.best_params_)

pred_val_grid_knn = grid_knn.predict_proba(X_valid_best)[:, 1]
print("Gini (GridSearch KNN):", gini_score(y_valid, pred_val_grid_knn))

Best GridSearch params (KNN): {'metric': 'euclidean', 'n_neighbors': 11, 'weights': 'distance'}
Gini (GridSearch KNN): 0.36498746450050623


### RandomSearch

In [21]:
param_dist_knn = {
    "n_neighbors": np.arange(3, 21),
    "weights": ["uniform", "distance"],
    "metric": ["euclidean", "manhattan", "minkowski"]
}

random_knn = RandomizedSearchCV(
    KNeighborsClassifier(),
    param_distributions=param_dist_knn,
    n_iter=20,
    scoring="roc_auc",
    cv=3,
    n_jobs=-1,
    random_state=42
)

random_knn.fit(X_train[top_features], y_train)

pred_val_rand_knn = random_knn.predict_proba(X_valid[top_features])[:, 1]
print("Gini (RandomSearch KNN):", gini_score(y_valid, pred_val_rand_knn))

Gini (RandomSearch KNN): 0.3960817164652215


### Optuna

In [22]:
def objective_knn(trial):
    n_neighbors = trial.suggest_int("n_neighbors", 3, 20)
    weights = trial.suggest_categorical("weights", ["uniform", "distance"])
    metric = trial.suggest_categorical("metric", ["euclidean", "manhattan", "minkowski"])
    
    knn = KNeighborsClassifier(
        n_neighbors=n_neighbors,
        weights=weights,
        metric=metric
    )
    
    score = cross_val_score(knn, X_train[top_features], y_train, cv=3, scoring="roc_auc").mean()
    return score

study_knn = optuna.create_study(direction="maximize")
study_knn.optimize(objective_knn, n_trials=30)

best_params_knn = study_knn.best_params
print("Best Optuna params (KNN):", best_params_knn)

knn_optuna = KNeighborsClassifier(**best_params_knn)
knn_optuna.fit(X_train[top_features], y_train)
pred_val_opt_knn = knn_optuna.predict_proba(X_valid[top_features])[:, 1]
print("Gini (Optuna KNN):", gini_score(y_valid, pred_val_opt_knn))

[I 2026-01-25 18:01:29,203] A new study created in memory with name: no-name-e2fe1794-26bd-4b3b-a31b-6b59500d82f1
[I 2026-01-25 18:01:32,518] Trial 0 finished with value: 0.6717129856885827 and parameters: {'n_neighbors': 6, 'weights': 'uniform', 'metric': 'minkowski'}. Best is trial 0 with value: 0.6717129856885827.
[I 2026-01-25 18:01:35,519] Trial 1 finished with value: 0.6642818322691484 and parameters: {'n_neighbors': 5, 'weights': 'uniform', 'metric': 'minkowski'}. Best is trial 0 with value: 0.6717129856885827.
[I 2026-01-25 18:01:40,378] Trial 2 finished with value: 0.7010815483441014 and parameters: {'n_neighbors': 16, 'weights': 'uniform', 'metric': 'euclidean'}. Best is trial 2 with value: 0.7010815483441014.
[I 2026-01-25 18:01:45,889] Trial 3 finished with value: 0.6678316317313285 and parameters: {'n_neighbors': 5, 'weights': 'distance', 'metric': 'manhattan'}. Best is trial 2 with value: 0.7010815483441014.
[I 2026-01-25 18:01:50,856] Trial 4 finished with value: 0.69233

Best Optuna params (KNN): {'n_neighbors': 20, 'weights': 'distance', 'metric': 'euclidean'}
Gini (Optuna KNN): 0.3964968982550343


## Results

In [23]:
results = []

# Logistic Regression
results.append({
    "Model": "LogisticRegression",
    "Tuning Method": "GridSearch",
    "Gini": gini_score(y_valid, pred_val_grid_lr)
})
results.append({
    "Model": "LogisticRegression",
    "Tuning Method": "RandomSearch",
    "Gini": gini_score(y_valid, pred_val_rand_lr)
})
results.append({
    "Model": "LogisticRegression",
    "Tuning Method": "Optuna",
    "Gini": gini_score(y_valid, pred_val_opt_lr)
})

# Naive Bayes
results.append({
    "Model": "GaussianNB",
    "Tuning Method": "GridSearch",
    "Gini": gini_score(y_valid, pred_val_grid_nb)
})
results.append({
    "Model": "GaussianNB",
    "Tuning Method": "RandomSearch",
    "Gini": gini_score(y_valid, pred_val_rand_nb)
})
results.append({
    "Model": "GaussianNB",
    "Tuning Method": "Optuna",
    "Gini": gini_score(y_valid, pred_val_opt_nb)
})

# KNN
results.append({
    "Model": "KNN",
    "Tuning Method": "GridSearch",
    "Gini": gini_score(y_valid, pred_val_grid_knn)
})
results.append({
    "Model": "KNN",
    "Tuning Method": "RandomSearch",
    "Gini": gini_score(y_valid, pred_val_rand_knn)
})
results.append({
    "Model": "KNN",
    "Tuning Method": "Optuna",
    "Gini": gini_score(y_valid, pred_val_opt_knn)
})

# Таблица
df_results = pd.DataFrame(results)
df_results.sort_values(["Model", "Tuning Method"], inplace=True)
df_results


,Model,Tuning Method,Gini
3,GaussianNB,GridSearch,0.458917
5,GaussianNB,Optuna,0.458938
4,GaussianNB,RandomSearch,0.458919
6,KNN,GridSearch,0.364987
8,KNN,Optuna,0.396497
7,KNN,RandomSearch,0.396082
0,LogisticRegression,GridSearch,0.470963
2,LogisticRegression,Optuna,0.472881
1,LogisticRegression,RandomSearch,0.471497


In [26]:
joblib.dump(study.best_trial.params, '../data/07/best_params_lr.pkl')

['../data/07/best_params_lr.pkl']